In [ ]:
import numpy as np
from collections import defaultdict
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.csgraph import dijkstra
import igl

# ----------------------------
# Basic geometry utilities
# ----------------------------
def face_areas(V, F):
    v0, v1, v2 = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
    return 0.5 * np.linalg.norm(np.cross(v1 - v0, v2 - v0), axis=1)

def face_centroids(V, F):
    return (V[F[:, 0]] + V[F[:, 1]] + V[F[:, 2]]) / 3.0

def avg_radius(V, F, weight_by_area=True):
    """
    PSB: avgRadius = average Euclidean distance from a point on surface to centroid. :contentReference[oaicite:5]{index=5}
    这里用 face centroid 的面积加权近似（更贴近“surface point”）。
    """
    C = V.mean(axis=0)  # mesh centroid (vertex mean). PSB uses centroid; this is a common approximation.
    Fc = face_centroids(V, F)
    d = np.linalg.norm(Fc - C[None, :], axis=1)
    if weight_by_area:
        A = face_areas(V, F)
        s = A.sum()
        return float((d * A).sum() / s) if s > 0 else float(d.mean())
    return float(d.mean())

# ----------------------------
# Build mesh connectivity
# ----------------------------
def build_face_adjacency(F):
    """
    Returns:
      adj_pairs: (m,2) face index pairs sharing an undirected edge
      shared_edges: (m,2) vertex indices of that shared edge (sorted)
    """
    edge2faces = defaultdict(list)
    for fi, (a, b, c) in enumerate(F):
        edges = [(a, b), (b, c), (c, a)]
        for u, v in edges:
            if u > v:
                u, v = v, u
            edge2faces[(u, v)].append(fi)

    adj_pairs = []
    shared_edges = []
    for (u, v), faces in edge2faces.items():
        if len(faces) == 2:
            f0, f1 = faces
            adj_pairs.append((f0, f1))
            shared_edges.append((u, v))
    if len(adj_pairs) == 0:
        return np.zeros((0, 2), dtype=int), np.zeros((0, 2), dtype=int)
    return np.asarray(adj_pairs, dtype=int), np.asarray(shared_edges, dtype=int)

def build_vertex_graph(V, F):
    """
    Build sparse adjacency of vertices with edge weights = Euclidean length.
    Dijkstra on this approximates geodesic distance along mesh edges.
    """
    edges = set()
    for a, b, c in F:
        for u, v in [(a, b), (b, c), (c, a)]:
            if u > v:
                u, v = v, u
            edges.add((u, v))
    edges = np.array(list(edges), dtype=int)
    u = edges[:, 0]
    v = edges[:, 1]
    w = np.linalg.norm(V[u] - V[v], axis=1)

    # undirected -> add both directions
    I = np.concatenate([u, v])
    J = np.concatenate([v, u])
    W = np.concatenate([w, w])
    G = coo_matrix((W, (I, J)), shape=(V.shape[0], V.shape[0])).tocsr()
    return G

# ----------------------------
# Boundary extraction (cuts)
# ----------------------------
def boundary_edges_from_face_labels(F, labels):
    """
    Return boundary edges as (m,2) vertex indices (sorted).
    A mesh edge is boundary iff it has exactly two incident faces and their labels differ.
    """
    edge2faces = defaultdict(list)
    for fi, (a, b, c) in enumerate(F):
        for u, v in [(a,b),(b,c),(c,a)]:
            if u > v: u, v = v, u
            edge2faces[(u,v)].append(fi)

    boundary_edges = []
    for (u, v), faces in edge2faces.items():
        if len(faces) == 2:
            f0, f1 = faces
            if labels[f0] != labels[f1]:
                boundary_edges.append((u, v))
    if not boundary_edges:
        return np.zeros((0,2), dtype=int)
    return np.asarray(boundary_edges, dtype=int)

def boundary_vertices_from_face_labels(F, labels):
    """
    Boundary vertex set (for multi-source Dijkstra).
    """
    be = boundary_edges_from_face_labels(F, labels)
    if be.shape[0] == 0:
        return np.zeros((0,), dtype=int)
    return np.unique(be.reshape(-1))

def boundary_midpoints_from_face_labels(V, F, labels):
    """
    Optional: sample boundary as edge midpoints (more 'point-set like').
    Useful if later you switch to KDTree in Euclidean space (not geodesic).
    """
    be = boundary_edges_from_face_labels(F, labels)
    if be.shape[0] == 0:
        return np.zeros((0,3), dtype=float)
    return (V[be[:,0]] + V[be[:,1]]) * 0.5

# ----------------------------
# 1) Cut Discrepancy (CD)
# ----------------------------
def cut_discrepancy(V, F, labels1, labels2, normalize=True):
    """
    CD(S1,S2) = (DCD(S1->S2) + DCD(S2->S1)) / avgRadius  :contentReference[oaicite:7]{index=7}
    DCD approximated as mean shortest-path distance (vertex graph) from boundary vertices in one
    segmentation to nearest boundary vertex in the other segmentation (multi-source Dijkstra).
    """
    b1 = boundary_vertices_from_face_labels(F, labels1)
    b2 = boundary_vertices_from_face_labels(F, labels2)
    if b1.size == 0 or b2.size == 0:
        # PSB notes CD undefined when either has zero cuts. :contentReference[oaicite:8]{index=8}
        return np.nan

    G = build_vertex_graph(V, F)

    # multi-source shortest paths: distance to nearest boundary set
    # run dijkstra once per boundary-set by adding a "super source" is tricky in sparse;
    # easiest: run dijkstra with indices=b2 and then take min across sources.
    dist_from_b2 = dijkstra(G, directed=False, indices=b2, return_predecessors=False)
    # dist_from_b2 shape: (len(b2), nV) -> min over sources
    d1 = np.min(dist_from_b2[:, b1], axis=0).mean()

    dist_from_b1 = dijkstra(G, directed=False, indices=b1, return_predecessors=False)
    d2 = np.min(dist_from_b1[:, b2], axis=0).mean()

    cd = float(d1 + d2)
    if normalize:
        r = avg_radius(V, F, weight_by_area=True)
        cd = cd / r if r > 0 else cd
    return cd

# ----------------------------
# 2) Hamming Distance (HD) per PSB
# ----------------------------
def _contingency_by_face_area(V, F, labels1, labels2):
    """
    Build overlap matrix M where M[i,j] = total area of faces with label1=i and label2=j.
    """
    A = face_areas(V, F)
    u, inv_u = np.unique(labels1, return_inverse=True)
    v, inv_v = np.unique(labels2, return_inverse=True)
    n1, n2 = len(u), len(v)

    M = np.zeros((n1, n2), dtype=float)
    np.add.at(M, (inv_u, inv_v), A)
    area1 = M.sum(axis=1)  # area per seg in S1
    area2 = M.sum(axis=0)  # area per seg in S2
    total = float(A.sum())
    return M, area1, area2, total

def hamming_distance_psb(V, F, labels1, labels2):
    """
    PSB directional hamming:
      DH(S1 => S2) = sum_i |S2_i \ S1_best(i)|  with best correspondence maximizing overlap :contentReference[oaicite:9]{index=9}
    Then:
      Rm = DH(S1=>S2)/|S| , Rf = DH(S2=>S1)/|S| , HD = 0.5*(Rm+Rf)  :contentReference[oaicite:10]{index=10}
    Here |.| uses surface area (more faithful than face count).
    """
    M, area1, area2, total = _contingency_by_face_area(V, F, labels1, labels2)
    if total <= 0:
        return np.nan

    # DH(S1 => S2): for each segment in S2 (columns), find best match in S1 (row) maximizing overlap
    best_overlap_for_S2 = M.max(axis=0) if M.size else np.zeros_like(area2)
    DH_1_to_2 = float((area2 - best_overlap_for_S2).sum())

    # DH(S2 => S1): swap roles
    best_overlap_for_S1 = M.max(axis=1) if M.size else np.zeros_like(area1)
    DH_2_to_1 = float((area1 - best_overlap_for_S1).sum())

    Rm = DH_1_to_2 / total
    Rf = DH_2_to_1 / total
    HD = 0.5 * (Rm + Rf)
    return HD, Rm, Rf

# ----------------------------
# 3) Rand Index (RI)
# ----------------------------
def rand_index_face_labels(labels1, labels2, return_error_like_psb=True):
    """
    RI as in PSB is the standard Rand index over face pairs. :contentReference[oaicite:11]{index=11}
    Efficient computation via contingency table (O(n + k^2)), no explicit O(n^2) pairs.

    If return_error_like_psb=True, returns (1 - RI) to match PSB reporting. :contentReference[oaicite:12]{index=12}
    """
    labels1 = np.asarray(labels1)
    labels2 = np.asarray(labels2)
    n = labels1.size
    if n <= 1:
        return 0.0 if return_error_like_psb else 1.0

    # relabel to 0..k-1
    _, a = np.unique(labels1, return_inverse=True)
    _, b = np.unique(labels2, return_inverse=True)

    k1 = a.max() + 1
    k2 = b.max() + 1
    # contingency counts
    M = np.zeros((k1, k2), dtype=np.int64)
    np.add.at(M, (a, b), 1)

    def comb2(x):
        return x * (x - 1) // 2

    sum_ij = comb2(M).sum()
    sum_a = comb2(M.sum(axis=1)).sum()
    sum_b = comb2(M.sum(axis=0)).sum()
    total_pairs = comb2(n)

    # agreements = same-same + diff-diff
    # same-same = sum_ij
    # diff-diff = total_pairs - (pairs same in A + pairs same in B - same-same)
    agreements = sum_ij + (total_pairs - (sum_a + sum_b - sum_ij))
    RI = agreements / float(total_pairs)

    return (1.0 - RI) if return_error_like_psb else RI

# ----------------------------
# 4) Consistency Error (GCE / LCE)
# ----------------------------
def consistency_error_gce_lce(V, F, labels1, labels2, use_area=True):
    """
    Local refinement error:
      E(S1,S2,fi) = |R(S1,fi) \ R(S2,fi)| / |R(S1,fi)|  :contentReference[oaicite:13]{index=13}
    GCE = (1/n) * min( sum_i E(S1,S2,fi), sum_i E(S2,S1,fi) )  :contentReference[oaicite:14]{index=14}
    LCE = (1/n) * sum_i min(E(S1,S2,fi), E(S2,S1,fi))           :contentReference[oaicite:15]{index=15}

    PSB uses |.| as a "measure" (can be face count or area). :contentReference[oaicite:16]{index=16}
    Here you can choose use_area=True for area-weighted, else face-count.
    """
    labels1 = np.asarray(labels1)
    labels2 = np.asarray(labels2)
    nF = labels1.size
    if nF == 0:
        return np.nan, np.nan

    w = face_areas(V, F) if use_area else np.ones(nF, dtype=float)

    # For each face, we need:
    # |R1|, |R2|, and |R1 ∩ R2| where R1 is segment of face in S1, R2 in S2.
    # We can get these via area-weighted contingency table.
    _, a = np.unique(labels1, return_inverse=True)
    _, b = np.unique(labels2, return_inverse=True)
    k1 = a.max() + 1
    k2 = b.max() + 1

    M = np.zeros((k1, k2), dtype=float)  # weighted overlap
    np.add.at(M, (a, b), w)
    size1 = M.sum(axis=1)  # |R1|
    size2 = M.sum(axis=0)  # |R2|

    inter = M[a, b]        # |R1 ∩ R2| per-face (look up by its (seg1,seg2))
    s1 = size1[a]
    s2 = size2[b]

    # E(S1,S2,fi) = (|R1| - |R1∩R2|) / |R1|
    # E(S2,S1,fi) = (|R2| - |R1∩R2|) / |R2|
    eps = 1e-12
    E12 = (s1 - inter) / np.maximum(s1, eps)
    E21 = (s2 - inter) / np.maximum(s2, eps)

    # unweighted mean over faces per PSB: 1/n sum_i ...
    # if you want area-weighted average over surface points, you can replace with (w*E).sum()/w.sum().
    gce = min(E12.sum(), E21.sum()) / float(nF)
    lce = np.minimum(E12, E21).sum() / float(nF)
    return float(gce), float(lce)

# ----------------------------
# Convenience wrapper
# ----------------------------
def eval_all_metrics(V, F, labels_pred, labels_gt):
    cd = cut_discrepancy(V, F, labels_pred, labels_gt, normalize=True)
    hd, rm, rf = hamming_distance_psb(V, F, labels_pred, labels_gt)
    ri_err = rand_index_face_labels(labels_pred, labels_gt, return_error_like_psb=True)  # PSB reports 1-RI
    gce, lce = consistency_error_gce_lce(V, F, labels_pred, labels_gt, use_area=True)
    return {
        "CutDiscrepancy": cd,
        "HammingDistance": hd,
        "MissingRate_Rm": rm,
        "FalseAlarmRate_Rf": rf,
        "RandIndexError_1-RI": ri_err,
        "GCE": gce,
        "LCE": lce,
    }


In [ ]:
def read_obj(obj_path):
    v, f = igl.read_triangle_mesh(obj_path)
    return v, f

def read_labels(label_path):
    with open(label_path, 'r') as file:
        lines = file.readlines()
        labels = [int(line.strip()) for line in lines]
    return labels

In [4]:
models_name = [
    "00023435_385b221a0d58490b84f3edee_trimesh_000",
    "00023471_7f45ff9e8c754def8bb4b1cb_trimesh_000",
    "00023576_35c7024f831f44048104c331_trimesh_000",
    "00023582_9c917172a61b472fb0e6ae3c_trimesh_002",
    "00023792_30c31f050d2c40139e9b36ca_trimesh_001",
    "00024036_de448e53313b4faa9dff2245_trimesh_003",
    "00024040_de448e53313b4faa9dff2245_trimesh_007",
    "00024082_de448e53313b4faa9dff2245_trimesh_049",
    "00024132_a087bf84d26046349bbbd6f6_trimesh_000",
    "00024448_274a7c9d6def4a699961a747_trimesh_002",
    "00024792_34a17822747a4b20a8c2954b_trimesh_009",
    "00024961_e26f4b039c6c48c9878b21fe_trimesh_000",
    "00025063_9347be1a9dad4e70852dcce2_trimesh_000",
    "00025183_021c990acfc648618a49bd47_trimesh_000",
    '00140180_7b0779cf3270ccee61e01b3e_trimesh_000',
    "00140223_5374ce893909eddbc7536b71_trimesh_000",
    "00140302_dce0fb9e808b32ded73025c1_trimesh_000",
    "00140400_e4a5e7862a9beda824b2e00c_trimesh_003",
    "00140553_e2c0841b6c86e3bfdcc8c477_trimesh_000",
    '00140710_33073a6cb55fadd20ec0e3fb_trimesh_001',
    "00140750_0da4d13f288d4cd70deecf20_trimesh_001",
    "00140756_eaf6606df2a105f356a99677_trimesh_000",
    "00140919_3023c3cfd7d6b1a80700e17e_trimesh_000",
    "00140923_0e818d7ff8780682567c824e_trimesh_000",
    "00140936_9a2b76f7e489ea90f5e41ef8_trimesh_007",
    "00140967_631426bb5199fa39fc8eeb1c_trimesh_008",
    "00140982_f2e481e34cff3a450799acac_trimesh_000",
    "00141097_57222034e4b08229a097b208_trimesh_010",
    'cad16'
]

# model_name = '00140180_7b0779cf3270ccee61e01b3e_trimesh_000'



def eval_once(obj_path, labels_part2surf_path, labels_segmat_path, labels_samesh_path, labels_partfield_path, labels_our_path, labels_gt_path):
    v, f = read_obj(obj_path)
    labels_our = read_labels(labels_our_path)
    labels_part2surf = read_labels(labels_part2surf_path)
    labels_samesh = read_labels(labels_samesh_path)
    labels_partfield = read_labels(labels_partfield_path)
    labels_segmat = read_labels(labels_segmat_path)
    labels_gt = read_labels(labels_gt_path)
    
    # print("nF:", len(labels_our), len(labels_part2surf), len(labels_samesh), f.shape[0])
    # print("unique labels_our:", np.unique(labels_our).size, np.unique(labels_our))
    # print("unique labels_part2surf:", np.unique(labels_part2surf).size, np.unique(labels_part2surf))
    # print("unique labels_samesh:", np.unique(labels_samesh).size, np.unique(labels_samesh))
    
    metric_part2surf_gt = eval_all_metrics(v, f, labels_part2surf, labels_gt)
    metric_segmat_gt = eval_all_metrics(v, f, labels_segmat, labels_gt)
    metric_samesh_gt = eval_all_metrics(v, f, labels_samesh, labels_gt)
    metric_partfield_gt = eval_all_metrics(v, f, labels_partfield, labels_gt)
    metric_our_gt = eval_all_metrics(v, f, labels_our, labels_gt)

    print("Part2Surf", metric_part2surf_gt)
    print("SegMat   ", metric_segmat_gt)
    print("SameMesh ", metric_samesh_gt)
    print("PartField", metric_partfield_gt)
    print("Our      ", metric_our_gt)
    


def eval_all():
    for model_name in models_name:
        print(model_name)
        obj_path = f"example_data/final/selected/{model_name}.obj"
        labels_part2surf_path = f"example_data/final/selected/{model_name}_part2surf.seg"
        labels_samesh_path = f"example_data/final/selected/{model_name}_samesh.seg"
        labels_partfield_path = f"example_data/final/selected/{model_name}_partfield.seg"
        labels_segmat_path = f"example_data/final/selected/{model_name}_segmat.seg"
        labels_our_path = f"example_data/final/selected/{model_name}_our.seg"
        labels_gt_path = f"example_data/final/selected/{model_name}_gt.seg"
        eval_once(obj_path, labels_part2surf_path, labels_segmat_path, labels_samesh_path, labels_partfield_path, labels_our_path, labels_gt_path)
        print("--------------------------------------------------")
eval_all()

00023435_385b221a0d58490b84f3edee_trimesh_000
Part2Surf {'CutDiscrepancy': 0.006596572799182794, 'HammingDistance': 0.04351447107665906, 'MissingRate_Rm': 0.0, 'FalseAlarmRate_Rf': 0.08702894215331812, 'RandIndexError_1-RI': 0.008626629214037007, 'GCE': 0.0, 'LCE': 0.0}
SegMat    {'CutDiscrepancy': nan, 'HammingDistance': 0.42840189984031896, 'MissingRate_Rm': 0.0, 'FalseAlarmRate_Rf': 0.8568037996806379, 'RandIndexError_1-RI': 0.9235167098730527, 'GCE': 0.0, 'LCE': 0.0}
SameMesh  {'CutDiscrepancy': nan, 'HammingDistance': 0.42840189984031896, 'MissingRate_Rm': 0.0, 'FalseAlarmRate_Rf': 0.8568037996806379, 'RandIndexError_1-RI': 0.9235167098730527, 'GCE': 0.0, 'LCE': 0.0}
PartField {'CutDiscrepancy': nan, 'HammingDistance': 0.42840189984031896, 'MissingRate_Rm': 0.0, 'FalseAlarmRate_Rf': 0.8568037996806379, 'RandIndexError_1-RI': 0.9235167098730527, 'GCE': 0.0, 'LCE': 0.0}
Our       {'CutDiscrepancy': 0.0, 'HammingDistance': 0.0, 'MissingRate_Rm': 0.0, 'FalseAlarmRate_Rf': 0.0, 'RandIn